# 06b — Generalization across individual batches

Repeat the comparison on batches 1, 4, 5 and 6 with fixed representation,
thresholds, budget and shared seed-specific splits. This is internal validation
within PLAsTiCC, not validation on real Rubin alerts.

The experiment cell launches the AL runs; existing summary CSVs do not by
themselves make that cell a cache-only operation.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, time
from pathlib import Path
raiz = Path.cwd()
while not (raiz / 'config.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent
os.chdir(raiz); sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from src.models import classification
from src.models import anomalies as an
from src.models import active_learning as al
from src import pipeline
RS = config.RANDOM_STATE

BATCHES  = [1, 4, 5, 6]          
FRAC_REP = 0.6                   
SEEDS = tuple(range(10))      
print('batches:', BATCHES, '· frac_desc:', FRAC_REP, '· seeds:', len(SEEDS))


## 1. Training model and global metadata


In [ ]:
RECOMPUTE = False
RESULTS_PATH = config.DIR_PROCESSED / 'generalizacion_por_batch.csv'
if not RECOMPUTE and not RESULTS_PATH.exists():
    raise FileNotFoundError('Import saved multi-batch results or explicitly enable RECOMPUTE.')
if RECOMPUTE:
    tr = classification.features_e2gp('train')
    X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()
    clf0 = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
    meta_glob = pd.read_csv(config.TEST_METADATA_CSV, usecols=['object_id', 'true_target'])
    
    def make_model(y_lab):
        return classification.e2gp_al(y=y_lab, random_state=RS)
    print('classifier listo')


## 2. Run the three policies on each batch


In [ ]:
if RECOMPUTE:
    import warnings
    warnings.filterwarnings('ignore', message='.*sklearn\\.utils\\.parallel\\.delayed.*', category=UserWarning)
    def cargar_batch(b):
        batch_paths = pipeline.select_batch(b)
        te = classification.features_e2gp('test', batch=b).merge(meta_glob, on='object_id')
        tt = te['true_target'].to_numpy()
        X_te, _ = classification.e2gp_matrix(te, COLS)
        gp = te[['object_id']].merge(pd.read_csv(batch_paths.features_gp_test_csv), on='object_id', how='left')
        st = te[['object_id']].merge(pd.read_csv(batch_paths.features_test_csv), on='object_id', how='left')
        desc = np.column_stack([an.score_isolation_forest(X_te),
                                an.gp_residual_score(gp), an.amplitude_score(st)])
        con = tt < 990; v, c = np.unique(tt[con], return_counts=True); sing = set(v[c < 2].tolist())
        keep = ~(con & np.isin(tt, list(sing)))
        return X_te[keep], tt[keep], desc[keep]
    
    records_list = []
    for b in BATCHES:
        t = time.time()
        X_te, tt, desc = cargar_batch(b)
        n_anom = int((tt >= 990).sum())
        
        hdc = al.two_channel_multiseed(
            X_tr, y_tr, X_te, tt, desc, make_model, fracs=(FRAC_REP,), random_seeds=SEEDS,
            conf_dudoso=0.60, budget=1000, b_iter=100, n_iter=30, verbose=False)
        fdc = hdc.sort_values('iter').groupby('semilla').tail(1)
        for s, r in fdc.set_index('semilla').iterrows():
            records_list.append({'batch': b, 'sistema': 'two channels', 'semilla': s,
                          'MF1': r['MF1'], 'recall': r['recall_anom'], 'descub': r['n_descubiertas'], 'n_anom': n_anom})
        
        htr = al.confidence_triage_multiseed(
            X_tr, y_tr, X_te, tt, make_model, modes_list=('triage', 'random'), random_seeds=SEEDS,
            conf_anom=0.30, conf_dudoso=0.60, orden_dudoso='entropy',
            budget=1000, b_iter=100, n_iter=30, verbose=False)
        ftr = htr.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
        for (m, s), r in ftr.set_index(['modo', 'semilla']).iterrows():
            records_list.append({'batch': b, 'sistema': 'triage' if m == 'triage' else 'random', 'semilla': s,
                          'MF1': r['MF1'], 'recall': r['recall_anom'], 'descub': r['n_descubiertas'], 'n_anom': n_anom})
        print(f'batch {b}: n_anom={n_anom} · hecho in {(time.time()-t)/60:.1f} min')
    
    res = pd.DataFrame(records_list)
    res.to_csv(config.DIR_PROCESSED / 'generalizacion_por_batch.csv', index=False)
    print(f'\nguardado: generalizacion_by_batch.csv ({len(res)} rows)')
else:
    res = pd.read_csv(RESULTS_PATH)
    if res.duplicated(['batch','sistema','semilla']).any():
        raise ValueError('Duplicate multi-batch experiment rows')
    expected = {(b,s,seed) for b in BATCHES for s in ['two channels','triage','random'] for seed in SEEDS}
    if set(res[['batch','sistema','semilla']].itertuples(index=False,name=None)) != expected:
        raise ValueError('Incomplete batch/system/seed coverage')
    print('Reusing all 120 saved batch/system/seed results; no AL experiment repeated.')


## 3. Aggregate by batch and policy


In [ ]:
n = res['semilla'].nunique()
g = (res.groupby(['batch', 'sistema'])
     .agg(MF1=('MF1', 'mean'), MF1_ee=('MF1', lambda x: x.std()/np.sqrt(n)),
          recall=('recall', 'mean'), recall_ee=('recall', lambda x: x.std()/np.sqrt(n)),
          descub=('descub', 'mean')).round(4))
sort_order = ['two channels', 'triage', 'random']
tab = g.reset_index()
piv_mf1 = tab.pivot(index='batch', columns='sistema', values='MF1')[sort_order]
piv_rec = tab.pivot(index='batch', columns='sistema', values='recall')[sort_order]
print('Macro-F1 by batch and system:'); display(piv_mf1)
print('Anomaly recall by batch and system:'); display(piv_rec)


## 4. Plot batch-level performance


In [ ]:
sis = ['two channels', 'triage', 'random']
col = {'two channels': '#2166ac', 'triage': '#d6604d', 'random': '#9aa0a6'}
mk  = {'two channels': 'o', 'triage': 's', 'random': '^'}
bx = np.arange(len(BATCHES)); w = 0.25
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, metric, ylab in [(axes[0], 'MF1', 'Macro-F1 (known classes)'),
                         (axes[1], 'recall', 'Recall of anomalies (99x)')]:
    for j, s in enumerate(sis):
        d = g.loc[[(b, s) for b in BATCHES]]
        y = d[metric].values; e = d[metric + '_ee'].values
        ax.bar(bx + (j-1)*w, y, w, yerr=e, capsize=4, color=col[s], label=s)
    ax.set_xticks(bx); ax.set_xticklabels([f'batch {b}' for b in BATCHES])
    ax.set_ylabel(ylab); ax.grid(axis='y', alpha=.3); ax.legend(fontsize=8)
    ax.set_title('Classification' if metric == 'MF1' else 'Anomaly discovery')
fig.suptitle(f'Three systems across individual batches · two channels (frac={FRAC_REP}) · {n} seeds',
             fontweight='bold')
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / 'generalizacion_por_batch.png', dpi=150); plt.show()


## Interpretation

Compare both absolute performance and paired policy differences. The relative
benefit of a discovery quota depends on the batch. Variation across DDF and WFD
data should not be interpreted as an isolated causal effect of survey depth.
